# 03. Visualize the Graph

Prerequisite: run `01_ingest_and_deduplicate.ipynb` first so this notebook has a populated FalkorDB graph to query.

Notebook 2 is optional, but if you run it first this notebook can also draw a memory-centered subgraph.

Install prerequisites before running the notebook:

```bash
uv sync --extra falkordblite --extra notebooks --extra viz
```


In [ ]:
import math
from pathlib import Path

import autoroot  # noqa
import matplotlib.pyplot as plt
import networkx as nx
from dotenv import load_dotenv
from rich import print

from grawiki.db import FalkorGraphDB

load_dotenv()


def locate_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "experimental_data").exists():
        return cwd
    if (cwd / "notebooks" / "experimental_data").exists():
        return cwd / "notebooks"
    raise FileNotFoundError(
        "Could not locate the notebooks directory from the current working directory."
    )


NOTEBOOKS_DIR = locate_notebooks_dir()
DB_PATH = NOTEBOOKS_DIR / "local_falcor.db"
database = FalkorGraphDB(db_path=DB_PATH, graph_name="grawiki_notebooks")

In [ ]:
seed_rows = database.ro_query(
    "MATCH (e:__entity__) RETURN e.id, e.name, labels(e) ORDER BY e.semantic_key, e.id LIMIT 20"
).result_set
seed_ids = [str(row[0]) for row in seed_rows]

edge_rows = database.ro_query(
    "MATCH (source)-[rel]->(target) WHERE source.id IN $ids AND target.id IN $ids RETURN source.id, source.name, labels(source), target.id, target.name, labels(target), type(rel) ORDER BY source.name, type(rel), target.name",
    {"ids": seed_ids},
).result_set

if not edge_rows:
    edge_rows = database.ro_query(
        "MATCH (source)-[rel]-(target) WHERE source.id IN $ids OR target.id IN $ids RETURN source.id, source.name, labels(source), target.id, target.name, labels(target), type(rel) ORDER BY source.name, type(rel), target.name LIMIT 30",
        {"ids": seed_ids},
    ).result_set

print({"seed_count": len(seed_rows), "edge_count": len(edge_rows)})

In [ ]:
def node_family(labels) -> str:
    labels = set(labels)
    for system_label in ("__memory__", "__chunk__", "__document__", "__entity__"):
        if system_label in labels:
            return system_label
    return "other"


def build_graph(rows):
    graph = nx.Graph()
    for (
        source_id,
        source_name,
        source_labels,
        target_id,
        target_name,
        target_labels,
        rel_label,
    ) in rows:
        graph.add_node(
            source_id,
            name=source_name,
            labels=list(source_labels),
            family=node_family(source_labels),
        )
        graph.add_node(
            target_id,
            name=target_name,
            labels=list(target_labels),
            family=node_family(target_labels),
        )
        graph.add_edge(source_id, target_id, label=rel_label)
    return graph


def draw_graph(graph, title: str, *, draw_edge_labels: bool = True):
    color_map = {
        "__entity__": "#5B8FF9",
        "__memory__": "#F6BD16",
        "__chunk__": "#61DDAA",
        "__document__": "#65789B",
        "other": "#BFBFBF",
    }
    if graph.number_of_nodes() == 0:
        print(f"{title}: graph is empty.")
        return

    plt.figure(figsize=(12, 8))
    k = 1.4 / math.sqrt(max(graph.number_of_nodes(), 1))
    pos = nx.spring_layout(graph, seed=44, k=k)
    node_colors = [
        color_map.get(graph.nodes[node]["family"], color_map["other"])
        for node in graph.nodes
    ]
    labels = {node: graph.nodes[node]["name"][:50] for node in graph.nodes}

    nx.draw_networkx_nodes(
        graph, pos, node_color=node_colors, node_size=1200, alpha=0.95
    )
    nx.draw_networkx_edges(graph, pos, width=1.8, alpha=0.7)
    nx.draw_networkx_labels(graph, pos, labels=labels, font_size=9)

    if draw_edge_labels and graph.number_of_edges() <= 20:
        edge_labels = {(u, v): data["label"] for u, v, data in graph.edges(data=True)}
        nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=8)

    plt.title(title)
    plt.axis("off")
    plt.show()


entity_graph = build_graph(edge_rows)
draw_graph(
    entity_graph,
    "Entity-focused graph view",
    draw_edge_labels=entity_graph.number_of_edges() <= 12,
)

## Optional focused views

The next cells show two smaller views: one entity ego graph and one memory-centered graph when memory nodes exist.


In [ ]:
if not seed_ids:
    print("No entity seeds were found.")
else:
    focus_id = seed_ids[0]
    focus_rows = database.ro_query(
        "MATCH (seed:__entity__ {id: $id})-[rel]-(neighbor) RETURN seed.id, seed.name, labels(seed), neighbor.id, neighbor.name, labels(neighbor), type(rel) ORDER BY neighbor.name, type(rel) LIMIT 12",
        {"id": focus_id},
    ).result_set
    focus_graph = build_graph(focus_rows)
    if focus_graph.number_of_nodes() == 0:
        print("No relationships were found for the selected entity seed.")
    else:
        draw_graph(
            focus_graph,
            f"Ego graph around {focus_graph.nodes[focus_id]['name']}",
            draw_edge_labels=True,
        )

In [ ]:
memory_rows = database.ro_query(
    "MATCH (m:__memory__) RETURN m.id, m.name ORDER BY m.creation_date, m.id LIMIT 1"
).result_set

if not memory_rows:
    print(
        "No memory nodes found. Run notebook 02 first if you want a memory-centered view."
    )
else:
    memory_id = str(memory_rows[0][0])
    subgraph_rows = database.ro_query(
        "MATCH (seed:__memory__ {id: $id})-[rel]-(neighbor) RETURN seed.id, seed.name, labels(seed), neighbor.id, neighbor.name, labels(neighbor), type(rel) ORDER BY neighbor.name, type(rel) LIMIT 20",
        {"id": memory_id},
    ).result_set
    memory_graph = build_graph(subgraph_rows)
    if memory_graph.number_of_nodes() == 0:
        print("The selected memory has no drawable neighbors in this query window.")
    else:
        draw_graph(
            memory_graph,
            f"Memory-centered view: {memory_graph.nodes[memory_id]['name']}",
            draw_edge_labels=memory_graph.number_of_edges() <= 12,
        )

In [ ]:
# Run this once when you are finished with the notebook session.
database.close()